In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/phanhuycng/tiktok-mini/Data.xlsx
/kaggle/input/datasets/phanhuycng/vi-hsd/__huggingface_repos__.json
/kaggle/input/datasets/phanhuycng/vi-hsd/ViHSD_jsonl/test.jsonl
/kaggle/input/datasets/phanhuycng/vi-hsd/ViHSD_jsonl/validation.jsonl
/kaggle/input/datasets/phanhuycng/vi-hsd/ViHSD_jsonl/train.jsonl
/kaggle/input/datasets/phanhuycng/bitsandbtyes/offline_wheels/shellingham-1.5.4-py2.py3-none-any.whl
/kaggle/input/datasets/phanhuycng/bitsandbtyes/offline_wheels/requests-2.34.2-py3-none-any.whl
/kaggle/input/datasets/phanhuycng/bitsandbtyes/offline_wheels/nvidia_cusparselt_cu13-0.8.1-py3-none-manylinux2014_x86_64.whl
/kaggle/input/datasets/phanhuycng/bitsandbtyes/offline_wheels/typing_extensions-4.15.0-py3-none-any.whl
/kaggle/input/datasets/phanhuycng/bitsandbtyes/offline_wheels/httpx-0.28.1-py3-none-any.whl
/kaggle/input/datasets/phanhuycng/bitsandbtyes/offline_wheels/nvidia_cufft-12.0.0.61-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl
/kaggle/input/datas

In [2]:
# ==========================================
# 0. CÀI ĐẶT OFFLINE (CHẠY ĐẦU TIÊN)
# ==========================================
# Thay đổi đường dẫn theo tên dataset bạn đã lưu
WHEELS_DIR = "/kaggle/input/datasets/phanhuycng/bitsandbtyes/offline_wheels"

# Cài đặt không cần mạng
!pip install --no-index --find-links={WHEELS_DIR} bitsandbytes accelerate datasets

Looking in links: /kaggle/input/datasets/phanhuycng/bitsandbtyes/offline_wheels
Processing /kaggle/input/datasets/phanhuycng/bitsandbtyes/offline_wheels/bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl


In [3]:


# ==========================================
# 1. IMPORT VÀ ĐỊNH NGHĨA ĐƯỜNG DẪN
# ==========================================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import load_dataset
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

MODEL_PATH = "/kaggle/input/models/phanhuycng/qwen-2-5-7b-instruct/pytorch/default/1/qwen-2.5-7b-instruct"


In [4]:
# ==========================================
# 2. LOAD TOKENIZER & MODEL
# ==========================================

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
print("Done!")

Loading tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading model...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Done!


In [5]:
# Load TikTok dataset
import pandas as pd

df_tiktok = pd.read_excel(
    '/kaggle/input/datasets/phanhuycng/tiktok-mini/Data.xlsx',
    sheet_name='threads',
)

# bỏ missing text
df_tiktok = df_tiktok.dropna(
    subset=['text']
).reset_index(drop=True)

# ép text về string
df_tiktok['text'] = (
    df_tiktok['text']
    .astype(str)
    .str.strip()
    .str.lower()
)

# bỏ text rỗng
df_tiktok = df_tiktok[
    df_tiktok['text'] != ""
].reset_index(drop=True)

print(f"TikTok dataset: {len(df_tiktok)} samples")

print(df_tiktok.shape)

print(df_tiktok["text"].apply(type).value_counts())

TikTok dataset: 20459 samples
(20459, 4)
text
<class 'str'>    20459
Name: count, dtype: int64


In [6]:
# Đếm số dòng có text kết thúc bằng "translate"
mask_translate = (
    df_tiktok['text']
    .astype(str)
    .str.strip()
    .str.endswith('translate')
)

num_translate = mask_translate.sum()

print(f"Số dòng kết thúc bằng 'translate': {num_translate}")


# Lưu before để xem ví dụ
before_samples = df_tiktok.loc[mask_translate, 'text'].copy()


# Xóa "translate" ở cuối câu
df_tiktok.loc[mask_translate, 'text'] = (
    df_tiktok.loc[mask_translate, 'text']
    .astype(str)
    .str.replace(r'\s*translate\s*$', '', regex=True)
    .str.strip()
)


# After samples
after_samples = df_tiktok.loc[mask_translate, 'text']


# In vài ví dụ before/after
print("\n===== BEFORE / AFTER SAMPLES =====\n")

for before, after in zip(before_samples.head(10), after_samples.head(10)):
    print("BEFORE:", repr(before))
    print("AFTER : ", repr(after))
    print("-" * 60)

Số dòng kết thúc bằng 'translate': 17704

===== BEFORE / AFTER SAMPLES =====

BEFORE: 'chuẩn bị thuốc mê với mấy con dao bạn ạ 🫣\n còn nghiêm túc thì bạn cứ đặt câu hỏi thôi : thích đội bóng nào? đội ấy có cầu thủ nào giỏi? vì sao thích? bạn chơi vị trí gì ...\n trong bóng đá gọi là chuyền bóng kiến tạo đấy ^^ , kiến tạo cho ngta nói về sở thích \n translate'
AFTER :  'chuẩn bị thuốc mê với mấy con dao bạn ạ 🫣\n còn nghiêm túc thì bạn cứ đặt câu hỏi thôi : thích đội bóng nào? đội ấy có cầu thủ nào giỏi? vì sao thích? bạn chơi vị trí gì ...\n trong bóng đá gọi là chuyền bóng kiến tạo đấy ^^ , kiến tạo cho ngta nói về sở thích'
------------------------------------------------------------
BEFORE: 'làm bóng cho nó đá đi b \n translate'
AFTER :  'làm bóng cho nó đá đi b'
------------------------------------------------------------
BEFORE: 'lỡ nó k chịu thì s b \n translate'
AFTER :  'lỡ nó k chịu thì s b'
------------------------------------------------------------
BEFORE: 'mổ bụng nó ra rồ

In [7]:
# ==========================================
# 4. ĐỊNH NGHĨA SYSTEM PROMPT & HÀM PREDICT
# ==========================================
SYSTEM_PROMPT = """
Bạn là hệ thống gán nhãn Hate Speech Detection cho bình luận mạng xã hội tiếng Việt.

Nhiệm vụ:
Đọc bình luận và gán CHÍNH XÁC MỘT nhãn duy nhất:
- [LABEL: 0]
- [LABEL: 1]
- [LABEL: 2]

====================
QUY TẮC GÁN NHÃN
====================

LABEL 0 = CLEAN

Dùng khi:
- Bình luận bình thường, trung tính.
- Có teencode, viết tắt, tiếng lóng nhưng không xúc phạm.
- Có ý kiến tiêu cực, lời khuyên, than phiền hoặc phán xét xã hội nhưng không công kích trực tiếp ai.
- Không có chửi thề nặng hoặc quấy rối rõ ràng.

Ví dụ:

Bình luận:
"team mình tới chưa mn"

Target cụ thể: Không
Công kích trực tiếp: Không
[LABEL: 0]

Bình luận:
"giờ yêu sớm nhiều khi khổ thật"

Target cụ thể: Không
Công kích trực tiếp: Không
[LABEL: 0]

Bình luận:
"bữa nay mệt ghê luôn á"

Target cụ thể: Không
Công kích trực tiếp: Không
[LABEL: 0]

====================

LABEL 1 = OFFENSIVE

Dùng khi:
- Có chửi thề, từ tục, toxic chung chung.
- Chửi đổng hoặc bộc lộ cảm xúc mạnh.
- KHÔNG nhắm rõ một cá nhân hoặc nhóm cụ thể.

Ví dụ:

Bình luận:
"clm chán vc"

Target cụ thể: Không
Công kích trực tiếp: Không
[LABEL: 1]

Bình luận:
"đúng xàm lol luôn"

Target cụ thể: Không
Công kích trực tiếp: Không
[LABEL: 1]

Bình luận:
"ngu vc thật"

Target cụ thể: Không
Công kích trực tiếp: Không
[LABEL: 1]

====================

LABEL 2 = HATE / TARGETED HARASSMENT

CHỈ dùng khi THỎA ĐỒNG THỜI:
1. Có đối tượng cụ thể:
   - cá nhân cụ thể
   - đại từ trực tiếp như "mày", "thằng", "con kia"
   - hoặc nhóm người/tôn giáo/quốc gia cụ thể

VÀ

2. Có công kích/xúc phạm/thù ghét rõ ràng nhắm vào đối tượng đó.

Nếu thiếu một trong hai điều kiện trên => KHÔNG được gán LABEL 2.

Ví dụ:

Bình luận:
"mày bị ngu hả"

Target cụ thể: Có
Công kích trực tiếp: Có
[LABEL: 2]

Bình luận:
"thằng đó đúng loại não tàn"

Target cụ thể: Có
Công kích trực tiếp: Có
[LABEL: 2]

Bình luận:
"bọn đó toàn lũ rác rưởi"

Target cụ thể: Có
Công kích trực tiếp: Có
[LABEL: 2]

====================
QUY TẮC QUAN TRỌNG
====================
- Đã xuất hiện từ bậy, thô tục, mất lịch sự ở dạng rõ thì chỉ có thể là OFFENSIVE hoặc HATE tùy theo ngữ cảnh câu
- KHÔNG coi lời khuyên, than phiền, kể chuyện là hate speech.
- KHÔNG coi teencode hoặc viết tắt là toxic nếu không có xúc phạm.
- Chỉ dùng LABEL 2 khi có công kích trực tiếp rõ ràng.

====================
ĐỊNH DẠNG OUTPUT
====================

Dòng 1:
Target cụ thể: Có hoặc Không

Dòng 2:
Công kích trực tiếp: Có hoặc Không

Dòng cuối:
[LABEL: số]
"""


In [8]:


def build_messages(text):
    """Dùng chat format của Qwen, không cần ghép string thủ công."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Bình luận: '{text}'"},
    ]


def parse_label(raw):
    """Tìm [LABEL: X] ở cuối output — đúng format model được yêu cầu."""
    import re
    matches = re.findall(r'\[LABEL:\s*([012])\]', raw)
    if matches:
        return int(matches[-1])   # lấy cái cuối cùng nếu model lặp lại
    # fallback: tìm chữ số đơn lẻ
    for ch in reversed(raw):
        if ch in ['0', '1', '2']:
            return int(ch)
    return -1


def predict_one(text, max_new_tokens=200):
    messages = build_messages(text)
    # Qwen hỗ trợ apply_chat_template với add_generation_prompt
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt",
                       truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output[0][inputs['input_ids'].shape[1]:]
    decoded   = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return decoded



In [9]:
torch.cuda.empty_cache()

In [10]:
# ==========================================
# 5. INFERENCE THEO BATCH (FULL DATAFRAME)
# ==========================================
import torch
from tqdm import tqdm

BATCH_SIZE = 64
tokenizer.padding_side = 'left'

def predict_batch(texts, max_new_tokens=200):
    prompts = [
        tokenizer.apply_chat_template(
            build_messages(text),
            tokenize=False,
            add_generation_prompt=True,
        )
        for text in texts
    ]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    results = []

    for i, output in enumerate(outputs):
        input_len = inputs['attention_mask'][i].sum()

        generated = output[input_len:]
        decoded = tokenizer.decode(
            generated,
            skip_special_tokens=True
        ).strip()

        results.append(decoded)

    return results


# ==========================================
# CHẠY MODEL TRÊN TOÀN BỘ df_tiktok
# ==========================================

texts_to_predict = df_tiktok['text'].astype(str).tolist()

print(f"Tổng số mẫu cần chạy model: {len(texts_to_predict)}")

predicted_labels = []
raw_outputs = []

for i in tqdm(range(0, len(texts_to_predict), BATCH_SIZE), desc="Inferencing"):
    batch_texts = texts_to_predict[i : i + BATCH_SIZE]

    raws = predict_batch(batch_texts)
    preds = [parse_label(r) for r in raws]

    raw_outputs.extend(raws)
    predicted_labels.extend(preds)


# ==========================================
# GÁN KẾT QUẢ VÀO DATAFRAME
# ==========================================

df_tiktok['predicted_label'] = predicted_labels

# ==========================================
# THỐNG KÊ
# ==========================================

unparsed = sum(1 for p in predicted_labels if p == -1)

print(f"\nDone! Unparsed: {unparsed}/{len(predicted_labels)}")


# Xem thử vài dòng
df_tiktok[['text', 'predicted_label']].head(10)

Tổng số mẫu cần chạy model: 20459


Inferencing: 100%|██████████| 320/320 [34:01<00:00,  6.38s/it]


Done! Unparsed: 0/20459


,text,predicted_label
0,chuẩn bị thuốc mê với mấy con dao bạn ạ 🫣\n cò...,2
1,làm bóng cho nó đá đi b,1
2,lỡ nó k chịu thì s b,1
3,mổ bụng nó ra rồi lấy thôi,1
4,lòng bạn khác lòng mình🫸❤️,0
5,làm fan mc,0
6,có bị đấm k b,0
7,nếu ở gần thì có thể xem cổ vũ xong đưa nước v...,0
8,"cảm ơn yk của b, nhưng chỗ đấy mình không đứng...",0
9,"xem nó chơi hnao cầm chai nước lạnh ra, đi coi...",2


In [11]:
# ==========================================
# XEM CÁC SAMPLE BỊ UNPARSED (-1)
# ==========================================

bad_idx = [i for i, p in enumerate(predicted_labels) if p == -1]

print(f"Tổng unparsed: {len(bad_idx)}")

for n, i in enumerate(bad_idx[:50], 1):

    print("\n" + "=" * 120)
    print(f"UNPARSED SAMPLE #{n}")
    print("=" * 120)

    print("\n[TEXT]")
    print(df_tiktok.iloc[i]["text"])

    print("\n[TRUE LABEL]")
    true_lbl = true_labels[i]
    print(f"{true_lbl} ({label_map.get(true_lbl, 'UNK')})")

    print("\n[RAW OUTPUT]")
    print(repr(raw_outputs[i]))

    print("\n[PREDICTED]")
    print(predicted_labels[i])

Tổng unparsed: 0


In [12]:
# ==========================================
# CHUYỂN TOÀN BỘ -1 -> 0 (CLEAN)
# ==========================================

predicted_labels = [
    0 if p == -1 else p
    for p in predicted_labels
]

# kiểm tra lại
from collections import Counter

label_map = {
    0: "CLEAN",
    1: "OFFENSIVE",
    2: "HATE"
}

pred_counter = Counter(predicted_labels)

print("\n===== FINAL PRED LABEL DISTRIBUTION =====")

for k in sorted(pred_counter.keys()):
    print(f"{k} ({label_map.get(k, 'UNK')}): {pred_counter[k]}")


===== FINAL PRED LABEL DISTRIBUTION =====
0 (CLEAN): 18664
1 (OFFENSIVE): 1338
2 (HATE): 457


In [13]:
import pandas as pd


# Đường dẫn lưu
output_path = '/kaggle/working/threads_predicted.xlsx'

# Xuất excel
df_tiktok.to_excel(output_path, index=False)

print(f"\nĐã lưu thành công file kết quả tại: {output_path}")


Đã lưu thành công file kết quả tại: /kaggle/working/threads_predicted.xlsx
